# GridLock Hackathon 2.0 — Traffic Demand Prediction
## Complete ML Pipeline (v3 — OOF encoding, tuned LightGBM submission)

**Target:** `max(0, 100 x R2)` on demand predictions  
**Dataset:** 77,299 train rows · 41,778 test rows · 10 features  
**Models:** LightGBM · XGBoost · CatBoost · Ensemble · Optuna (100 trials)

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import KFold
from scipy.optimize import minimize
import lightgbm as lgb
from lightgbm import LGBMRegressor, early_stopping as lgb_early_stop, log_evaluation as lgb_log
import xgboost as xgb
from catboost import CatBoostRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import pygeohash as pgh

plt.style.use('ggplot')
pd.set_option('display.max_columns', 50)
np.random.seed(42)
print('All imports successful!')

## Load Data

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
print(f'Train: {train.shape}  |  Test: {test.shape}')
print('\nTrain dtypes:\n', train.dtypes)
train.head(3)

---
## Step 1: Exploratory Data Analysis

### 1a. Demand Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(train['demand'], bins=60, edgecolor='black', alpha=0.75, color='steelblue')
axes[0].set_title('Demand Distribution', fontsize=13)
axes[0].set_xlabel('Demand'); axes[0].set_ylabel('Frequency')

axes[1].hist(np.log1p(train['demand']), bins=60, edgecolor='black', alpha=0.75, color='darkorange')
axes[1].set_title('log1p(Demand) Distribution', fontsize=13)
axes[1].set_xlabel('log1p(Demand)'); axes[1].set_ylabel('Frequency')

plt.suptitle('Traffic Demand Distribution', fontsize=15)
plt.tight_layout()
plt.savefig('demand_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nDemand Statistics:')
print(train['demand'].describe())
print(f'\nRows with demand > 0.5: {(train["demand"] > 0.5).sum()} ({100*(train["demand"] > 0.5).mean():.1f}%)')

### 1b. Unique Values

In [ ]:
for col in ['day', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']:
    vals = train[col].unique()
    print(f'{col:15s} ({len(vals)} unique): {vals}')
ts_unique = sorted(train['timestamp'].unique())
print(f'\ntimestamp: {len(ts_unique)} unique values | sample: {ts_unique[:8]}')

### 1c. Missing Values

In [ ]:
print('Missing values - TRAIN:')
miss_tr = train.isnull().sum()
print(miss_tr[miss_tr > 0])
print(f'\nTotal train missing: {train.isnull().sum().sum()}')

print('\nMissing values - TEST:')
miss_te = test.isnull().sum()
print(miss_te[miss_te > 0])
print(f'\nTotal test missing: {test.isnull().sum().sum()}')

### 1d. Correlation with Demand

In [ ]:
print('Pearson correlation with demand:')
for col in ['NumberofLanes', 'Temperature']:
    corr = train[col].corr(train['demand'])
    print(f'  {col:<20s}: {corr:+.4f}')

tmp = train.copy()
tmp['hour'] = tmp['timestamp'].apply(lambda x: int(x.split(':')[0]))
hourly = tmp.groupby('hour')['demand'].mean()

fig, ax = plt.subplots(figsize=(12, 4))
hourly.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Mean Demand by Hour of Day')
ax.set_xlabel('Hour'); ax.set_ylabel('Mean Demand')
plt.tight_layout()
plt.savefig('demand_by_hour.png', dpi=100, bbox_inches='tight')
plt.show()

### 1e. Mean Demand by Category

In [ ]:
for col in ['RoadType', 'Weather', 'LargeVehicles', 'Landmarks', 'NumberofLanes']:
    print(f'\nMean demand by {col}:')
    print(train.groupby(col, dropna=False)['demand'].agg(['mean','std','count']).round(4))

---
## Step 2: Feature Engineering

**v3 changes:** demand-based geohash encodings moved to OOF step (no leakage).  
`_transform` only adds structural/temporal/distributional features.  
OOF target encoding applied as a separate step after FE.

In [ ]:
class FeatureEngineer:
    'Fits on train, applies to test with zero leakage (v3 — no demand encoding in _transform).'

    @staticmethod
    def _decode_gh(gh):
        try:
            lat, lng, lat_err, lng_err = pgh.decode_exactly(gh)
            return float(lat), float(lng), float(lat_err), float(lng_err)
        except Exception:
            return 0.0, 0.0, 0.0, 0.0

    @staticmethod
    def _parse_ts(ts):
        h, m = ts.split(':')
        return int(h), int(m)

    @staticmethod
    def _time_of_day(h):
        if   h <= 5:  return 0
        elif h <= 11: return 1
        elif h <= 17: return 2
        else:         return 3

    def fit_transform(self, df):
        df = df.copy()

        # Imputation stats
        self._temp_med_global = float(df['Temperature'].median())
        self._temp_med_gh     = df.groupby('geohash')['Temperature'].median()
        self._rt_mode_global  = (df['RoadType'].mode().iloc[0]
                                 if len(df['RoadType'].mode()) else 'Residential')
        self._rt_mode_gh      = df.groupby('geohash')['RoadType'].apply(
            lambda x: x.mode().iloc[0] if len(x.mode()) else self._rt_mode_global)
        self._wx_mode_global  = (df['Weather'].mode().iloc[0]
                                 if len(df['Weather'].mode()) else 'Sunny')
        self._wx_mode_gh      = df.groupby('geohash')['Weather'].apply(
            lambda x: x.mode().iloc[0] if len(x.mode()) else self._wx_mode_global)

        df = self._impute(df)

        # Encoding params
        self._max_day = int(df['day'].max())
        self._rt_cols = sorted(df['RoadType'].dropna().unique().tolist())
        self._wx_cols = sorted(df['Weather'].dropna().unique().tolist())

        # Distributional geohash stats (kept — fit on full train)
        gb = df.groupby('geohash')['demand']
        self._gh_std   = gb.std().fillna(0)
        self._gh_p25   = gb.quantile(0.25)
        self._gh_p75   = gb.quantile(0.75)
        self._gh_max   = gb.max()
        self._gh_count = gb.count().astype(float)
        self._gh_mean  = gb.mean()  # stored for neighbor stats only
        self._g_mean   = float(df['demand'].mean())
        self._g_std    = float(df['demand'].std())

        # Global time stats (aggregated across all geohashes — low leakage)
        ts_tmp = df['timestamp'].apply(self._parse_ts)
        df['_hour'] = ts_tmp.apply(lambda x: x[0])
        df['_tod']  = df['_hour'].apply(self._time_of_day)
        self._hour_mean = df.groupby('_hour')['demand'].mean()
        self._hour_std  = df.groupby('_hour')['demand'].std().fillna(0)
        self._tod_mean  = df.groupby('_tod' )['demand'].mean()
        df = df.drop(columns=['_hour', '_tod'])

        # Neighbor geohash demand (second-order, low leakage)
        gh_mean_d = self._gh_mean.to_dict()
        self._neighbor_mean_d = {}
        for gh in df['geohash'].unique():
            try:
                nbrs = list(pgh.neighbors(gh).values())
                self._neighbor_mean_d[gh] = float(np.mean(
                    [gh_mean_d.get(n, self._g_mean) for n in nbrs]))
            except Exception:
                self._neighbor_mean_d[gh] = self._g_mean

        # Geohash prefix label encoders
        gh4 = df['geohash'].str[:4]
        gh5 = df['geohash'].str[:5]
        self._gh4_labels = {v: i for i, v in enumerate(sorted(gh4.unique()))}
        self._gh5_labels = {v: i for i, v in enumerate(sorted(gh5.unique()))}

        # Temperature bin edges (5 equal-width bins, fixed from train)
        _, self._temp_bin_edges = pd.cut(df['Temperature'], bins=5, retbins=True)
        self._temp_bin_edges[0]  = -np.inf
        self._temp_bin_edges[-1] =  np.inf

        # Demand rank per geohash (percentile rank of each geohash by mean demand)
        self._gh_rank = self._gh_mean.rank(pct=True).to_dict()

        self.fitted = True
        return self._transform(df)

    def transform(self, df):
        assert self.fitted, 'Call fit_transform first'
        df = df.copy()
        df = self._impute(df)
        return self._transform(df)

    def _impute(self, df):
        df['Temperature'] = df['Temperature'].fillna(
            df['geohash'].map(self._temp_med_gh).fillna(self._temp_med_global))
        df['RoadType'] = df['RoadType'].fillna(
            df['geohash'].map(self._rt_mode_gh).fillna(self._rt_mode_global))
        df['Weather'] = df['Weather'].fillna(
            df['geohash'].map(self._wx_mode_gh).fillna(self._wx_mode_global))
        return df

    def _transform(self, df):
        # Timestamp features
        ts = df['timestamp'].apply(self._parse_ts)
        df['hour']         = ts.apply(lambda x: x[0])
        df['minute']       = ts.apply(lambda x: x[1])
        df['hour_sin']     = np.sin(2 * np.pi * df['hour']   / 24)
        df['hour_cos']     = np.cos(2 * np.pi * df['hour']   / 24)
        df['minute_sin']   = np.sin(2 * np.pi * df['minute'] / 60)
        df['minute_cos']   = np.cos(2 * np.pi * df['minute'] / 60)
        df['is_rush_hour'] = df['hour'].isin([7,8,9,17,18,19]).astype(int)
        df['time_of_day']  = df['hour'].apply(self._time_of_day)

        # Geohash decoding
        geo = df['geohash'].apply(self._decode_gh)
        df['geohash_lat']     = geo.apply(lambda x: x[0])
        df['geohash_lng']     = geo.apply(lambda x: x[1])
        df['geohash_lat_err'] = geo.apply(lambda x: x[2])
        df['geohash_lng_err'] = geo.apply(lambda x: x[3])

        # Day features
        df['day_sin']   = np.sin(2 * np.pi * df['day'] / self._max_day)
        df['day_cos']   = np.cos(2 * np.pi * df['day'] / self._max_day)
        df['day_mod_7'] = df['day'] % 7

        # Categorical encoding
        df['LargeVehicles'] = (df['LargeVehicles'] == 'Allowed').astype(int)
        df['Landmarks']     = (df['Landmarks']     == 'Yes').astype(int)
        for rt in self._rt_cols:
            df[f'RoadType_{rt}'] = (df['RoadType'] == rt).astype(int)
        for wx in self._wx_cols:
            df[f'Weather_{wx}']  = (df['Weather']  == wx).astype(int)

        # Geohash distributional stats (from full train — minor leakage for train rows)
        df['geohash_std_demand'] = df['geohash'].map(self._gh_std  ).fillna(self._g_std)
        df['geohash_p25_demand'] = df['geohash'].map(self._gh_p25  ).fillna(self._g_mean * 0.4)
        df['geohash_p75_demand'] = df['geohash'].map(self._gh_p75  ).fillna(self._g_mean * 1.6)
        df['geohash_max_demand'] = df['geohash'].map(self._gh_max  ).fillna(self._g_mean * 2.5)
        df['geohash_count']      = df['geohash'].map(self._gh_count).fillna(1.0)

        # Neighbor demand (second-order, low leakage)
        df['neighbor_mean_demand'] = df['geohash'].map(self._neighbor_mean_d).fillna(self._g_mean)

        # Global time stats (no leakage — aggregate over all geohashes)
        df['hour_mean_demand'] = df['hour'].map(self._hour_mean).fillna(self._g_mean)
        df['hour_std_demand']  = df['hour'].map(self._hour_std ).fillna(self._g_std)
        df['tod_mean_demand']  = df['time_of_day'].map(self._tod_mean).fillna(self._g_mean)

        # Geohash hierarchy (strings for OOF keys + label-encoded ints as model features)
        df['geohash_4'] = df['geohash'].str[:4]
        df['geohash_5'] = df['geohash'].str[:5]
        df['geohash_4_label'] = df['geohash_4'].map(self._gh4_labels).fillna(-1).astype(int)
        df['geohash_5_label'] = df['geohash_5'].map(self._gh5_labels).fillna(-1).astype(int)

        # Temperature bins (5 equal-width from train distribution)
        df['temp_binned'] = pd.cut(
            df['Temperature'], bins=self._temp_bin_edges,
            labels=False, include_lowest=True
        ).fillna(2).astype(int)

        # Demand rank of geohash (percentile rank, no per-row demand used)
        df['demand_rank_in_geohash'] = df['geohash'].map(self._gh_rank).fillna(0.5)

        # Interaction features
        rt_num = {'Highway': 3, 'Street': 2, 'Residential': 1}
        df['RoadType_numeric'] = df['RoadType'].map(rt_num).fillna(0).astype(float)
        df['lanes_x_roadtype'] = df['NumberofLanes'] * df['RoadType_numeric']

        wx_num = {'Sunny': 1, 'Cloudy': 2, 'Rainy': 3, 'Foggy': 4, 'Snowy': 5}
        df['Weather_numeric'] = df['Weather'].map(wx_num).fillna(1).astype(float)
        df['temp_x_weather']  = df['Temperature'] * df['Weather_numeric']

        # hour x geohash_lat interaction
        df['hour_x_geohash_lat'] = df['hour'] * df['geohash_lat']

        return df

print('FeatureEngineer class defined (v3 — no demand encoding in _transform).')

### Apply Feature Engineering to Train and Test

In [ ]:
fe       = FeatureEngineer()
train_fe = fe.fit_transform(train)
test_fe  = fe.transform(test)

print(f'Train FE shape : {train_fe.shape}')
print(f'Test  FE shape : {test_fe.shape}')
print(f'\nSample engineered columns:')
train_fe[['geohash_lat','geohash_lng','hour','time_of_day',
          'geohash_4_label','geohash_5_label','temp_binned',
          'geohash_count','demand_rank_in_geohash','demand']].head(3)

### OOF Target Encoding (5-fold, no leakage)

For each train row, the geohash mean demand is computed from **out-of-fold** rows only.  
Test rows use full-train smoothed means (no target info available).  
Smoothing: `(count * mean + α * global_mean) / (count + α)`

In [ ]:
def oof_encode(train_df, test_df, group_cols, col_name,
               target='demand', n_splits=5, alpha=10):
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    gm = float(train_df[target].mean())
    n  = len(train_df)

    # Time-sort so KFold respects temporal order
    sort_key   = (train_df['day'] * 1440 + train_df['hour'] * 60 + train_df['minute']).values
    sort_pos   = np.argsort(sort_key)
    unsort_pos = np.argsort(sort_pos)
    df_s = train_df.iloc[sort_pos].reset_index(drop=True)

    enc_sorted = np.full(n, gm, dtype=float)
    kf = KFold(n_splits=n_splits, shuffle=False)

    for tr_idx, va_idx in kf.split(np.arange(n)):
        stats = df_s.iloc[tr_idx].groupby(group_cols)[target].agg(['mean', 'count'])
        stats['smooth'] = (stats['count'] * stats['mean'] + alpha * gm) / (stats['count'] + alpha)
        d = stats['smooth'].to_dict()
        if len(group_cols) == 1:
            keys = df_s.iloc[va_idx][group_cols[0]].tolist()
        else:
            keys = list(zip(*[df_s.iloc[va_idx][c].tolist() for c in group_cols]))
        enc_sorted[va_idx] = [d.get(k, gm) for k in keys]

    train_df[col_name] = enc_sorted[unsort_pos]

    # Full train smoothed stats for test (no OOF needed)
    full = train_df.groupby(group_cols)[target].agg(['mean', 'count'])
    full['smooth'] = (full['count'] * full['mean'] + alpha * gm) / (full['count'] + alpha)
    full_d = full['smooth'].to_dict()
    if len(group_cols) == 1:
        te_keys = test_df[group_cols[0]].tolist()
    else:
        te_keys = list(zip(*[test_df[c].tolist() for c in group_cols]))
    test_df[col_name] = [full_d.get(k, gm) for k in te_keys]

    return full_d, gm

print('oof_encode defined.')

In [ ]:
print('Applying OOF target encoding (this runs 5-fold x 6 encodings)...')

oof_encode(train_fe, test_fe, 'geohash',                 'geohash_mean_demand')
print('  1/6 geohash_mean_demand done')

oof_encode(train_fe, test_fe, ['geohash', 'hour'],       'geohash_hour_mean_demand')
print('  2/6 geohash_hour_mean_demand done')

oof_encode(train_fe, test_fe, ['geohash', 'time_of_day'],'geohash_timeofday_mean_demand')
print('  3/6 geohash_timeofday_mean_demand done')

oof_encode(train_fe, test_fe, 'geohash_4',               'geohash_4_mean_demand')
print('  4/6 geohash_4_mean_demand done')

oof_encode(train_fe, test_fe, 'geohash_5',               'geohash_5_mean_demand')
print('  5/6 geohash_5_mean_demand done')

oof_encode(train_fe, test_fe, 'temp_binned',             'temp_binned_mean_demand')
print('  6/6 temp_binned_mean_demand done')

print(f'\nOOF encoding complete.')
print(f'Train columns: {len(train_fe.columns)}  |  Test columns: {len(test_fe.columns)}')

### Post-OOF Derived Features

In [ ]:
for df in [train_fe, test_fe]:
    df['location_time']     = df['geohash_mean_demand']      * df['hour_sin']
    df['demand_vs_hour']    = df['geohash_mean_demand']      / (df['hour_mean_demand'] + 1e-9)
    df['gh_hour_vs_global'] = df['geohash_hour_mean_demand'] / (df['hour_mean_demand'] + 1e-9)

print('Derived interaction features added to train_fe and test_fe.')
print(f'Final train_fe shape : {train_fe.shape}')
print(f'Final test_fe  shape : {test_fe.shape}')

print('\nOOF feature sample (first 3 rows):')
train_fe[['geohash','geohash_mean_demand','geohash_hour_mean_demand',
          'geohash_4_mean_demand','temp_binned_mean_demand',
          'demand_rank_in_geohash','demand']].head(3)

### Define Feature Columns

In [ ]:
EXCLUDE = {
    'Index', 'geohash', 'timestamp', 'demand',
    'RoadType', 'Weather',
    'geohash_4', 'geohash_5'   # string prefix columns — labels used instead
}
feature_cols = [c for c in train_fe.columns if c not in EXCLUDE]
print(f'Total features: {len(feature_cols)}')
print('\nFeature list:')
for i, c in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {c}')

In [ ]:
X_all = train_fe[feature_cols].values.astype(np.float32)
y_all = train_fe['demand'].values.astype(np.float32)

nan_cnt = int(np.isnan(X_all).sum())
if nan_cnt > 0:
    nan_per_col = pd.DataFrame(X_all, columns=feature_cols).isnull().sum()
    print('NaN columns:', nan_per_col[nan_per_col > 0].to_dict())
    X_all = np.nan_to_num(X_all, nan=0.0)
    print(f'Replaced {nan_cnt} NaN values with 0')
else:
    print('No NaN values in X_all.')

print(f'X_all: {X_all.shape}  |  y_all: {y_all.shape}')
print(f'y range: [{y_all.min():.4f}, {y_all.max():.4f}]  mean: {y_all.mean():.4f}')

---
## Step 3: Modelling

**Time-aware split:** sort by day + time, hold out the **last 15%** as validation.

### 3a. Time-Aware Train / Validation Split

In [ ]:
sort_keys  = train_fe['day'] * 1440 + train_fe['hour'] * 60 + train_fe['minute']
sorted_idx = sort_keys.argsort().values

split_idx = int(len(sorted_idx) * 0.85)
tr_idx    = sorted_idx[:split_idx]
va_idx    = sorted_idx[split_idx:]

X_train, y_train = X_all[tr_idx], y_all[tr_idx]
X_val,   y_val   = X_all[va_idx], y_all[va_idx]

print(f'Train split : {X_train.shape}')
print(f'Val   split : {X_val.shape}')
print(f'Val demand  : mean={y_val.mean():.4f}  std={y_val.std():.4f}')

### 3b-1. LightGBM

In [ ]:
lgbm_model = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=255,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    n_jobs=-1,
    random_state=42,
    verbose=-1
)

lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb_early_stop(50, verbose=False), lgb_log(200)]
)

lgbm_pred_val = lgbm_model.predict(X_val)
lgbm_r2       = r2_score(y_val, lgbm_pred_val)
lgbm_rmse     = np.sqrt(mean_squared_error(y_val, lgbm_pred_val))

print(f'LightGBM  |  Val R2 = {lgbm_r2:.6f}  |  RMSE = {lgbm_rmse:.6f}')
print(f'Hackathon score estimate: {max(0, lgbm_r2 * 100):.2f}')
print(f'Best iteration: {lgbm_model.best_iteration_}')

lgbm_imp = pd.DataFrame({'feature': feature_cols,
                          'importance': lgbm_model.feature_importances_}
                        ).sort_values('importance', ascending=False)
print('\nTop 20 features (LightGBM):')
print(lgbm_imp.head(20).to_string(index=False))

### 3b-2. XGBoost

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    early_stopping_rounds=50,
    eval_metric='rmse',
    n_jobs=-1,
    random_state=42,
    verbosity=0
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

xgb_pred_val = xgb_model.predict(X_val)
xgb_r2       = r2_score(y_val, xgb_pred_val)
xgb_rmse     = np.sqrt(mean_squared_error(y_val, xgb_pred_val))

print(f'XGBoost   |  Val R2 = {xgb_r2:.6f}  |  RMSE = {xgb_rmse:.6f}')
print(f'Hackathon score estimate: {max(0, xgb_r2 * 100):.2f}')
print(f'Best iteration: {xgb_model.best_iteration}')

### 3b-3. CatBoost

In [ ]:
cat_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    min_data_in_leaf=20,
    subsample=0.8,
    colsample_bylevel=0.8,
    l2_leaf_reg=3.0,
    early_stopping_rounds=50,
    random_state=42,
    verbose=100
)

cat_model.fit(X_train, y_train, eval_set=(X_val, y_val))

cat_pred_val = cat_model.predict(X_val)
cat_r2       = r2_score(y_val, cat_pred_val)
cat_rmse     = np.sqrt(mean_squared_error(y_val, cat_pred_val))

print(f'CatBoost  |  Val R2 = {cat_r2:.6f}  |  RMSE = {cat_rmse:.6f}')
print(f'Hackathon score estimate: {max(0, cat_r2 * 100):.2f}')
print(f'Best iteration: {cat_model.best_iteration_}')

### 3c. Ensemble (comparison only — submission uses tuned LightGBM)

In [ ]:
preds_list = [lgbm_pred_val, xgb_pred_val, cat_pred_val]

# Fixed weights
ens_pred_val = 0.4*lgbm_pred_val + 0.3*xgb_pred_val + 0.3*cat_pred_val
ens_r2       = r2_score(y_val, ens_pred_val)

# Optimized weights via scipy
def neg_r2(w):
    w = np.abs(w) / np.abs(w).sum()
    return -r2_score(y_val, sum(w[i]*preds_list[i] for i in range(3)))

res   = minimize(neg_r2, x0=[0.4, 0.3, 0.3], method='Nelder-Mead',
                 options={'maxiter': 2000, 'xatol': 1e-6, 'fatol': 1e-6})
opt_w = np.abs(res.x) / np.abs(res.x).sum()
opt_ens_pred = sum(opt_w[i]*preds_list[i] for i in range(3))
opt_ens_r2   = r2_score(y_val, opt_ens_pred)

print('=' * 55)
print('Model comparison (val R2):')
print(f'  LightGBM              : {lgbm_r2:.6f}  ({max(0,lgbm_r2*100):.2f})')
print(f'  XGBoost               : {xgb_r2:.6f}  ({max(0,xgb_r2*100):.2f})')
print(f'  CatBoost              : {cat_r2:.6f}  ({max(0,cat_r2*100):.2f})')
print(f'  Fixed ensemble        : {ens_r2:.6f}  ({max(0,ens_r2*100):.2f})')
print(f'  Optimized ensemble    : {opt_ens_r2:.6f}  ({max(0,opt_ens_r2*100):.2f})')
print(f'  Opt weights: LGBM={opt_w[0]:.3f}  XGB={opt_w[1]:.3f}  CAT={opt_w[2]:.3f}')
print('=' * 55)
print('NOTE: submission will use tuned LightGBM only.')

---
## Step 4: Hyperparameter Tuning — LightGBM (100 Optuna trials)

Always tune LightGBM. Submission will use this tuned model on full training data.

In [ ]:
def objective_lgbm(trial):
    params = dict(
        n_estimators      = 2000,
        learning_rate     = trial.suggest_float('learning_rate',    0.01,  0.15,  log=True),
        num_leaves        = trial.suggest_int(  'num_leaves',       63,    511),
        min_child_samples = trial.suggest_int(  'min_child_samples',5,     100),
        subsample         = trial.suggest_float('subsample',        0.5,   1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5,   1.0),
        reg_alpha         = trial.suggest_float('reg_alpha',        1e-3,  10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda',       1e-3,  10.0, log=True),
        n_jobs=-1, random_state=42, verbose=-1
    )
    m = LGBMRegressor(**params)
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)],
          callbacks=[lgb_early_stop(50, verbose=False)])
    return r2_score(y_val, m.predict(X_val))

study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective_lgbm, n_trials=100, show_progress_bar=True)

print(f'\nBest trial R2 : {study.best_value:.6f}  ({max(0, study.best_value*100):.2f})')
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

### Retrain Tuned LightGBM on Full Training Data

In [ ]:
best_params = study.best_params.copy()
n_iters = int(lgbm_model.best_iteration_ * 1.2) + 1

print(f'Retraining tuned LightGBM on {X_all.shape[0]} rows ({n_iters} trees)...')

tuned_model = LGBMRegressor(
    n_estimators=n_iters,
    **best_params,
    n_jobs=-1, random_state=42, verbose=-1
)
tuned_model.fit(X_all, y_all)

tuned_pred_val = tuned_model.predict(X_val)
tuned_r2       = r2_score(y_val, tuned_pred_val)
print(f'\nTuned LightGBM (val reference): R2 = {tuned_r2:.6f}  |  Score = {max(0, tuned_r2*100):.2f}')

---
## Step 5: Final Predictions

Use **tuned LightGBM only** for submission (ensemble not used).  
Test features already have OOF encodings applied (full-train smoothed stats).

In [ ]:
X_test = test_fe[feature_cols].values.astype(np.float32)
print(f'Test features shape : {X_test.shape}')

nan_count = int(np.isnan(X_test).sum())
if nan_count > 0:
    nan_cols = pd.DataFrame(X_test, columns=feature_cols).isnull().sum()
    print('NaN columns:', nan_cols[nan_cols > 0].to_dict())
    X_test = np.nan_to_num(X_test, nan=0.0)
    print(f'Replaced {nan_count} NaN values with 0')
else:
    print('No NaN values in X_test.')

In [ ]:
max_train_demand = float(train['demand'].max())
print(f'Max training demand : {max_train_demand:.6f}')

# Always use tuned LightGBM for submission
final_preds = tuned_model.predict(X_test)
final_preds = np.clip(final_preds, 0.0, max_train_demand)

print(f'\nFinal prediction stats (tuned LightGBM):')
print(f'  min  : {final_preds.min():.6f}')
print(f'  max  : {final_preds.max():.6f}')
print(f'  mean : {final_preds.mean():.6f}')
print(f'  std  : {final_preds.std():.6f}')
print(f'  pct zeros: {100*(final_preds == 0).mean():.1f}%')

In [ ]:
submission = pd.DataFrame({'Index': test['Index'], 'demand': final_preds})
assert submission.shape == (41778, 2), f'Shape mismatch: {submission.shape}'
submission.to_csv('submission.csv', index=False)

print(f'submission.csv saved  |  shape: {submission.shape}')
print('\nFirst 5 rows:')
print(submission.head())
print('\nPredictions describe():')
print(submission['demand'].describe())

---
## Step 6: Validation Report

In [ ]:
print('=' * 65)
print('  GRIDLOCK HACKATHON 2.0 - VALIDATION REPORT (v3 OOF)')
print('=' * 65)

models_summary = {
    'LightGBM (baseline)' : lgbm_r2,
    'XGBoost  (baseline)' : xgb_r2,
    'CatBoost (baseline)' : cat_r2,
    'Ensemble fixed'      : ens_r2,
    'Ensemble optimized'  : opt_ens_r2,
    'Tuned LightGBM'      : tuned_r2,
}

print('\n{:<28s} {:>10s}  {:>8s}'.format('Model', 'Val R2', 'Score'))
print('-' * 55)
for name, r2 in sorted(models_summary.items(), key=lambda x: -x[1]):
    score  = max(0, r2 * 100)
    marker = '  <- SUBMISSION' if name == 'Tuned LightGBM' else ''
    print(f'  {name:<26s} {r2:.6f}  {score:6.2f}{marker}')

print(f'\nSubmission model  : Tuned LightGBM (full data, {n_iters} trees)')
print(f'Estimated score   : {max(0, tuned_r2 * 100):.2f} / 100')
print(f'Total features    : {len(feature_cols)}')

print('\nTop 10 Most Important Features (LightGBM baseline):')
for _, row in lgbm_imp.head(10).iterrows():
    print(f'  {row["feature"]:<35s} {row["importance"]:.2f}')

print(f'\nSubmission file  : submission.csv')
print(f'Submission shape : {submission.shape}')
print('=' * 65)